In [ ]:
 # install libraries
#!pip install datasets
#!pip install --upgrade transformers[torch]
#!pip install --upgrade torch
#!pip install --upgrade accelerate

In [2]:
# import libraries

# main libs
import numpy as np
import pandas as pd
from tqdm import tqdm
import random
import os
import warnings

# data
from datasets import load_dataset

# sklearn for evaluations
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score

# transformers
import torch
import torch.nn as nn
import transformers
import accelerate
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer # models & tokenisers
from transformers import TrainingArguments, Trainer # load training libs
from transformers import EarlyStoppingCallback # to set early stopping

In [3]:
# disable WANDB_MODE -- important!
import os
os.environ["WANDB_MODE"] = "disabled"

In [4]:
# Suppress the warning
warnings.filterwarnings("ignore",
                        message="Repo card metadata block was not found.",
                        category=UserWarning)

In [5]:
# check for GPU availability
torch.cuda.is_available()

True

In [6]:
# check correct import
print("Pytorch version:", torch.__version__) # pytorch 2.1.0+

Pytorch version: 2.14.1+cu130


In [7]:
# Set device
device = torch.device('cpu') # cpu for windows, mps for mac
print(device)

cpu


In [8]:
# Mount drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True) # To force: force_remount=True

Mounted at /content/drive


## Load data

We will use the same datasets of reviews, used in previous weeks.

In [9]:
# import the Hugging Face datasets library (pip install datasets)
from datasets import load_dataset

# download the political tweets dataset from the Hugging Face Hub
dataset = load_dataset("Jacobvs/PoliticalTweets")

# the dataset can contain several splits; take the first one
split_name = list(dataset.keys())[0]

# convert that split into a pandas DataFrame
df_labelled = dataset[split_name].to_pandas()

README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

formatted_data.parquet: reconstructing file:   0%|          |  0.00B / 26.6MB            

formatted_data.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/190491 [00:00<?, ? examples/s]

In [32]:
# remove possible duplicate tweets & subset
df_labelled = df_labelled.drop_duplicates(subset=['text'], keep='first')
df_labelled = df_labelled[['text']] # subset
df_labelled

,text
0,Happy th birthday to the @USNavy! The strength...
1,The greatest generation's investment in infras...
2,"Thanks to @SenTedCruz and @SenatorWarnock, th..."
3,/ To get lasting change we cant just lock up t...
4,Today were celebrating years of the Hoosier st...
...,...
190486,Religious liberty is foundational to America. ...
190487,"During #NationalPoliceWeek, I want to give a h..."
190488,The same week that gas prices hit a record hig...
190489,Amid another mounting crisis—a nationwide shor...


In [33]:
# set text as string
df_labelled['text'] = df_labelled['text'].astype(str)

In [14]:
# optional: the dataset is large, uncomment to train on a random subset for a faster run
# df_labelled = df_labelled.sample(n=5000, random_state=seed)

In [36]:
# safety copy
df = df_labelled[:100]
df

,text
0,Happy th birthday to the @USNavy! The strength...
1,The greatest generation's investment in infras...
2,"Thanks to @SenTedCruz and @SenatorWarnock, th..."
3,/ To get lasting change we cant just lock up t...
4,Today were celebrating years of the Hoosier st...
...,...
95,President Biden has presided over a disastrous...
96,Im disappointed @SenateDems reversed their lon...
97,Some history: in the last century governments ...
98,The stepped-up basis loophole provides a huge ...


## Apply Classifier

In [20]:
# set pretrained model path
repo_id = "Sami92/mmbert-is-political"

In [21]:
# load pre-trained model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(repo_id)
model = AutoModelForSequenceClassification.from_pretrained(repo_id)

config.json:   0%|          | 0.00/2.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.23GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

In [22]:
# add labels
labels = ['0', '1'] # 0=non-political, 1=political

In [37]:
# softmax
def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)

In [38]:
# loop function on text
for index, row in tqdm(df.iterrows()):
    tweet = row['text']

    # Calculate label
    encoded_input = tokenizer(tweet,
                              truncation=True,
                              padding=True,
                              return_tensors="pt") #max_length = 144
    output = model(**encoded_input)
    scores = output[0][0].detach().numpy()
    scores = softmax(scores)

    ranking = np.argsort(scores)
    ranking = ranking[::-1]
    for i in range(scores.shape[0]):
        label = labels[ranking[i]]
        score = scores[ranking[i]]
        df.at[index, label] = np.round(
            float(score), 6)
    df.at[index, 'pred_label'] = labels[ranking[0]] #save new column w top label

0it [00:00, ?it/s]/tmp/ipykernel_302/3440253688.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.at[index, label] = np.round(
/tmp/ipykernel_302/3440253688.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.at[index, label] = np.round(
/tmp/ipykernel_302/3440253688.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/st

In [39]:
# quick check n. labels
df['pred_label'].value_counts()

,count
pred_label,
1,84
0,16


In [40]:
# quick check on text and labels
df.head(10)

,text,1,0,pred_label
0,Happy th birthday to the @USNavy! The strength...,1.000000,0.000000,1
1,The greatest generation's investment in infras...,1.000000,0.000000,1
2,"Thanks to @SenTedCruz and @SenatorWarnock, th...",1.000000,0.000000,1
3,/ To get lasting change we cant just lock up t...,1.000000,0.000000,1
4,Today were celebrating years of the Hoosier st...,0.000000,1.000000,0
5,The #ForthePeopleAct includes reforms that are...,1.000000,0.000000,1
6,"Todays strong, bipartisan vote is just the beg...",1.000000,0.000000,1
7,Supporting crime victims requires holding crim...,1.000000,0.000000,1
8,We in Louisiana know how natural disasters cha...,1.000000,0.000000,1
9,Today we start anew.,0.000141,0.999859,0


In [ ]:
# save new df
#df.to_csv("/content/drive/MyDrive/llm-demo-folder/df_coded_tweets.csv", ## change based on your drive
#               index=False)